## Transforming Data Using the Medallion Architecture

### ![image_1771980413506.png](./image_1771980413506.png "image_1771980413506.png")

In [0]:
%sql
USE CATALOG azuredatabricks;
USE SCHEMA default;

SELECT current_catalog(), current_schema();

In [0]:
spark.sql("LIST '/Volumes/azuredatabricks/default/myfiles'").display()

## B. Simple Example of the Medallion Architecture

**Objective:** Create a job that can be scheduled to run on a schedule. The pipeline will:

1. Ingest all CSV files from the **myfiles** volume and create a bronze table.  
2. Prepare the bronze table by adding new columns and create a silver table.  
3. Create a gold aggregated table for consumers.

## BRONZE

**Objective:** Create a table using all of the CSV files in the **myfiles** volume.

1. Execute the cell to perform the following steps:

   - The `DROP TABLE IF EXISTS` statement drops the **current_employees_bronze** table if it already exists (for demonstration purposes).

   - The `CREATE TABLE IF NOT EXISTS` statement creates the Delta table **current_employees_bronze** if it doesn't already exist and defines its columns.

   - The `COPY INTO` statement:
     - Loads all the CSV files from the **myfiles** volume in your schema into the **current_employees_bronze** table.
     - Uses the first row as headers and infers the schema from the CSV files.

   - The final `SELECT` query displays all rows from the **current_employees_bronze** table.

View the results and confirm that the table contains **6 rows and 4 columns**.

In [0]:
%sql
DROP TABLE current_employees_bronze;
DROP TABLE IF EXISTS current_employees_bronze;
CREATE TABLE current_employees_bronze (
  ID INT,
  Firstname STRING,
  Country STRING,
  Role STRING
);

In [0]:
%sql
SHOW TABLES;

In [0]:
# Create bronze raw ingestion table
spark.sql('''
          COPY INTO current_employees_bronze
          FROM '/Volumes/azuredatabricks/default/myfiles/'
          FILEFORMAT = CSV
          FORMAT_OPTIONS (
              'header' = 'true',
              'inferSchema' = 'true')
          ''').display()

In [0]:
%sql
SELECT * FROM current_employees_bronze;

# SILVER

**Objective:** Transform the bronze table and create the silver table.

1. Create a table named `current_employees_silver` from the `current_employees_bronze` table.

   The table will:
   - Select the columns `ID`, `FirstName`, `Country`.
   - Convert the `Role` column to uppercase.
   - Add two new columns: `CurrentTimeStamp` and `CurrentDate`.

In [0]:
%sql
DROP TABLE IF EXISTS current_employees_silver;

CREATE TABLE current_employees_silver
AS 
SELECT ID, Firstname, Country, upper(Role) AS Role, current_timestamp() AS timestamp_col, current_date() AS date_col
FROM current_employees_bronze;

In [0]:
%sql
SELECT * FROM current_employees_silver;

In [0]:
%sql
SHOW TABLES;

### GOLD

**Objective:** Aggregate the silver table to create the final gold table.

1. Create a temporary view named `temp_view_total_roles` that aggregates the total number of employees by role. Then, display the results of the view.

In [0]:
%sql
DROP TABLE IF EXISTS temp_view_total_roles;

CREATE OR REPLACE TEMP VIEW temp_view_total_roles
AS 
SELECT Role, COUNT(*) TotalEmployees
FROM current_employees_silver
GROUP BY Role;
    
SELECT * FROM temp_view_total_roles;


Create the final gold table named total_roles_gold with the specified columns.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS total_roles_gold (
  Role STRING,
  TotalEmployees INT
);

### 3. Insert Aggregated Rows into Gold Table

Insert all rows from the aggregated temporary view `temp_view_total_rows` into the `total_roles_gold` table, overwriting the existing data in the table.

> ⚠️ This operation overwrites the data in the table but keeps the existing schema, table definition, and properties.

#### ✅ Confirm the Following:
- `num_affected_rows` is **4**
- `num_inserted_rows` is **4**

In [0]:
%sql
INSERT OVERWRITE total_roles_gold 
SELECT * FROM temp_view_total_roles;

In [0]:
%sql
SELECT * FROM total_roles_gold;

In [0]:
%sql
DESCRIBE HISTORY total_roles_gold;

In [0]:
%sql
-- Drop the tables
DROP TABLE IF EXISTS current_employees_bronze;
DROP TABLE IF EXISTS current_employees_silver;
DROP TABLE IF EXISTS total_roles_gold;